In [1]:
import pandas as pd

df=pd.read_csv("uber_fare_dataset_with_nulls.csv")

In [2]:
df

,pickup_hour,day_of_week,weather_condition,pickup_zone,vehicle_type,distance_km,passenger_count,surge_multiplier,fare_amount
0,6,Tuesday,Foggy,University,Uber Black,1.23,4.0,1.00,6.93
1,19,Tuesday,Rainy,University,UberXL,15.49,4.0,1.84,50.78
2,14,Tuesday,Clear,Airport,UberX,1.00,2.0,NaN,4.26
3,10,Thursday,Snowy,Airport,UberX,8.50,3.0,1.30,21.15
4,7,Wednesday,Clear,Mall,UberX,7.47,2.0,1.40,16.79
...,...,...,...,...,...,...,...,...,...
4995,11,Tuesday,Clear,University,UberX,26.73,NaN,1.00,38.35
4996,10,Saturday,Clear,Suburb,Uber Black,14.46,1.0,1.20,40.62
4997,14,Tuesday,Clear,Suburb,UberXL,1.90,1.0,1.00,5.25
4998,15,Thursday,Foggy,Airport,UberX,2.69,1.0,1.00,5.65


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pickup_hour        5000 non-null   int64  
 1   day_of_week        5000 non-null   object 
 2   weather_condition  4600 non-null   object 
 3   pickup_zone        4800 non-null   object 
 4   vehicle_type       5000 non-null   object 
 5   distance_km        4750 non-null   float64
 6   passenger_count    4700 non-null   float64
 7   surge_multiplier   4650 non-null   float64
 8   fare_amount        4850 non-null   float64
dtypes: float64(4), int64(1), object(4)
memory usage: 351.7+ KB


In [4]:
df.isnull().sum()

pickup_hour            0
day_of_week            0
weather_condition    400
pickup_zone          200
vehicle_type           0
distance_km          250
passenger_count      300
surge_multiplier     350
fare_amount          150
dtype: int64

In [5]:
df.fillna({'passenger_count': df['passenger_count'].median(),
          'surge_multiplier':df['surge_multiplier'].median(),
          'fare_amount':df['fare_amount'].median(),
          'distance_km':df['distance_km'].median()},inplace = True)

In [6]:
df.fillna({'weather_condition':df['weather_condition'].mode()[0],
          'pickup_zone':df['pickup_zone'].mode()[0]},inplace=True)

In [7]:
df.isnull().sum()

pickup_hour          0
day_of_week          0
weather_condition    0
pickup_zone          0
vehicle_type         0
distance_km          0
passenger_count      0
surge_multiplier     0
fare_amount          0
dtype: int64

In [8]:
df.drop_duplicates()
df.shape

(5000, 9)

In [9]:
df.drop(columns='pickup_zone',axis=1,inplace=True)
df.shape

(5000, 8)

In [10]:
X=df.drop(columns='fare_amount')
y=df['fare_amount']
print(X.shape)
print(y.shape)

(5000, 7)
(5000,)


# Model Training

In [11]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(4000, 7)
(4000,)
(1000, 7)
(1000,)


In [12]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pickup_hour        5000 non-null   int64  
 1   day_of_week        5000 non-null   object 
 2   weather_condition  5000 non-null   object 
 3   vehicle_type       5000 non-null   object 
 4   distance_km        5000 non-null   float64
 5   passenger_count    5000 non-null   float64
 6   surge_multiplier   5000 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 273.6+ KB


# Feature Scaling

In [16]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

transformer=ColumnTransformer(transformers=[("t1",OneHotEncoder(),[1,2,3]),
                                           ("t2",StandardScaler(),[0,4,5,6])])

X_train_trans=transformer.fit_transform(X_train)
X_test_trans=transformer.transform(X_test)



In [17]:
X_train_trans = pd.DataFrame(X_train_trans)
X_train_trans.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.360283,-0.731093,-0.480328,-0.258915
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.931408,1.557757,1.369756,-0.424865
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.927048,0.493854,0.444714,1.068683
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.788450,0.062713,-0.480328,-0.424865
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.498881,-0.781815,1.369756,-0.424865


In [19]:
ohe_cols = transformer.named_transformers_['t1'].get_feature_names_out()

num_cols = ['pickup_hour','distance_km','passenger_count','surge_multiplier']

all_cols = list(ohe_cols) + num_cols

X_train_trans.columns = all_cols

In [20]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

In [21]:
models = {
    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },
    
    "Ridge": {
        "model": Ridge(),
        "params": {
            "alpha": [0.1, 1, 10]
        }
    },
    
    "Lasso": {
        "model": Lasso(),
        "params": {
            "alpha": [0.001, 0.01, 0.1]
        }
    },
    
    "DecisionTree": {
        "model": DecisionTreeRegressor(),
        "params": {
            "max_depth": [5, 10, 20],
            "min_samples_split": [2, 5, 10]
        }
    },
    
    "RandomForest": {
        "model": RandomForestRegressor(),
        "params": {
            "n_estimators": [50, 100],
            "max_depth": [None, 10, 20]
        }
    },
    
    "GradientBoosting": {
        "model": GradientBoostingRegressor(),
        "params": {
            "n_estimators": [50, 100],
            "learning_rate": [0.01, 0.1]
        }
    }
}

In [22]:
best_models = {}

for name, mp in models.items():
    print(f"Running {name}...")
    
    grid = GridSearchCV(
        mp["model"],
        mp["params"],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    grid.fit(X_train_trans, y_train)
    
    best_models[name] = {
        "best_score": grid.best_score_,
        "best_params": grid.best_params_
    }

# Print results
for model, result in best_models.items():
    print(f"\n{model}")
    print("Best Score:", result["best_score"])
    print("Best Params:", result["best_params"])

Running LinearRegression...
Running Ridge...
Running Lasso...
Running DecisionTree...
Running RandomForest...
Running GradientBoosting...

LinearRegression
Best Score: -67.16106846883473
Best Params: {}

Ridge
Best Score: -67.156907408792
Best Params: {'alpha': 10}

Lasso
Best Score: -67.14801461052585
Best Params: {'alpha': 0.01}

DecisionTree
Best Score: -67.93842134971847
Best Params: {'max_depth': 10, 'min_samples_split': 10}

RandomForest
Best Score: -44.06361844882961
Best Params: {'max_depth': 10, 'n_estimators': 100}

GradientBoosting
Best Score: -39.74300120260274
Best Params: {'learning_rate': 0.1, 'n_estimators': 100}


In [23]:
best_model_name = max(best_models, key=lambda x: best_models[x]['best_score'])
print("Best Model:", best_model_name)

Best Model: GradientBoosting


In [24]:
best_model = GradientBoostingRegressor(
    learning_rate=0.1,
    n_estimators=100
)

best_model.fit(X_train_trans, y_train)

GradientBoostingRegressor()

In [25]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = best_model.predict(X_test_trans)

print("RMSE:", mean_squared_error(y_test, y_pred,))
print("R2 Score:", r2_score(y_test, y_pred))

RMSE: 24.23914473660218
R2 Score: 0.9260198617128625


C:\Users\Sai teja\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(


In [26]:
pip install xgboost lightgbm

In [27]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [28]:
models["XGBoost"] = {
    "model": XGBRegressor(random_state=42),
    "params": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 6]
    }
}

models["LightGBM"] = {
    "model": LGBMRegressor(random_state=42),
    "params": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "num_leaves": [31, 50]
    }
}

In [29]:
for name, mp in models.items():
    print(f"Running {name}...")
    
    grid = GridSearchCV(
        mp["model"],
        mp["params"],
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    
    grid.fit(X_train_trans, y_train)
    
    best_models[name] = {
        "best_score": grid.best_score_,
        "best_params": grid.best_params_
    }

Running LinearRegression...
Running Ridge...
Running Lasso...
Running DecisionTree...
Running RandomForest...
Running GradientBoosting...
Running XGBoost...
Running LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000677 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 334
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 19
[LightGBM] [Info] Start training from score 18.431828


In [31]:
from lightgbm import LGBMRegressor

models["LightGBM"] = {
    "model": LGBMRegressor(random_state=42, verbose=-1),
    "params": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1],
        "num_leaves": [31, 50]
    }
}

In [32]:
for model, result in best_models.items():
    print(f"\n{model}")
    print("Best Score:", result["best_score"])
    print("Best Params:", result["best_params"])


LinearRegression
Best Score: -67.16106846883473
Best Params: {}

Ridge
Best Score: -67.156907408792
Best Params: {'alpha': 10}

Lasso
Best Score: -67.14801461052585
Best Params: {'alpha': 0.01}

DecisionTree
Best Score: -70.25585085313281
Best Params: {'max_depth': 10, 'min_samples_split': 10}

RandomForest
Best Score: -44.09240999975678
Best Params: {'max_depth': 10, 'n_estimators': 100}

GradientBoosting
Best Score: -39.8074372770143
Best Params: {'learning_rate': 0.1, 'n_estimators': 100}

XGBoost
Best Score: -39.27516205132624
Best Params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}

LightGBM
Best Score: -42.30501721285528
Best Params: {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31}


In [33]:
from xgboost import XGBRegressor

final_model = XGBRegressor(
    learning_rate=0.1,
    max_depth=3,
    n_estimators=200,
    random_state=42
)

final_model.fit(X_train_trans, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [34]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = final_model.predict(X_test_trans)

print("RMSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

RMSE: 24.23327019162788
R2 Score: 0.9260377913739251


In [35]:
import os
import joblib

# set correct path
project_path = r"C:\Users\Sai teja\OneDrive\Desktop\Uber_Price_Prediction\models"

os.makedirs(project_path, exist_ok=True)

joblib.dump(final_model, os.path.join(project_path, "xgboost_model.pkl"))
joblib.dump(X_train.columns, os.path.join(project_path, "columns.pkl"))

print("Saved in correct folder ✅")

Saved in correct folder ✅


In [36]:
import os
print(os.getcwd())

C:\Users\Sai teja
